## Step 1: Create a Random Chromosome

In [141]:
import random

def create_chromosome(length):
    # Start with an empty list
    chrom = []
    # Add 'length' random bits (0 or 1)
    for i in range(length):
        bit = random.randint(0, 1)
        chrom.append(bit)
    return chrom

In [142]:
## Checkpoint 1: Run the following in your Python shell and verify the output.
c = create_chromosome(9)
print(c) # e.g. [1, 0, 0, 1, 1, 0, 0, 1, 0]
print(len(c))

[1, 1, 1, 0, 0, 0, 0, 0, 1]
9


## Step 2: Fitness Function

In [143]:
def fitness(chromosome):
    count = 0
    for gene in chromosome:
        if gene == 1:
            count = count + 1
    return count

In [144]:
## Checkpoint 2: Verify these results.
print(fitness([0,0,0,0,0,0,0,0,0])) # Expected: 0
print(fitness([1,1,1,1,1,1,1,1,1])) # Expected: 9
print(fitness([1,0,1,0,1,0,1,0,1]))

0
9
5


In [145]:
def select_parent(population, scores):
    # Step 1: Add up all fitness scores
    total = 0
    for s in scores:
        total = total + s

    # Step 2: Pick a random number between 0 and total
    spin = random.uniform(0, total)

    # Step 3: Walk through the population
    running_sum = 0
    for i in range(len(population)):
        running_sum = running_sum + scores[i]
        if spin <= running_sum:
            return population[i]

    # Fallback: return the last one
    return population[-1]

In [146]:
## Checkpoint 3: Test with this population:
pop = [[0,0,0,0,0,0,0,0,1], # fitness = 1
[1,1,1,1,1,1,1,1,0]] # fitness = 8
scores = [fitness(pop[0]), fitness(pop[1])]
## Run 100 selections—the second chromosome should be picked roughly 8 times more often.
count0 = 0
count1 = 0
for i in range(100):
    p = select_parent(pop, scores)
    if p == pop[0]:
        count0 = count0 + 1
    else:
        count1 = count1 + 1
print("First picked:", count0, "Second picked:", count1)

First picked: 10 Second picked: 90


## Step 4: Single-Point Crossover

In [147]:
def crossover(parent1, parent2):
    # Pick a random cut point (not at the very start or end)
    point = random.randint(1, len(parent1) - 1)

    # Build child1: left part of parent1 + right part of parent2
    child1 = parent1[:point] + parent2[point:]

    # Build child2: left part of parent2 + right part of parent1
    child2 = parent2[:point] + parent1[point:]

    return child1, child2

In [148]:
## Checkpoint 4: Manually verify with a fixed cut point.
p1 = [1,1,1,1,1,0,0,0,0]
p2 = [0,0,0,0,0,1,1,1,1]
c1 = p1[:5] + p2[5:]
c2 = p2[:5] + p1[5:]
print("Child 1:", c1) # [1,1,1,1,1,1,1,1,1]
print("Child 2:", c2) # [0,0,0,0,0,0,0,0,0]
## Question: What happens if the cut point is 1? What if it is 8?

Child 1: [1, 1, 1, 1, 1, 1, 1, 1, 1]
Child 2: [0, 0, 0, 0, 0, 0, 0, 0, 0]


## Step 5: Mutation

In [149]:
def mutate(chromosome, rate):
    new_chrom = []
    for gene in chromosome:
        # With small probability, flip the bit
        if random.random() < rate:
            if gene == 0:
                new_chrom.append(1)  # flip 0 to 1
            else:
                new_chrom.append(0)  # flip 1 to 0
        else:
            # No mutation, keep the gene as it is
            new_chrom.append(gene)
    return new_chrom

In [150]:
## Checkpoint 5: Test mutation behavior.
c = [0,0,0,0,0,0,0,0,0]
print(mutate(c, 1.0)) # rate=1.0: every bit flips → [1,1,1,1,1,1,1,1,1]
print(mutate(c, 0.0))

[1, 1, 1, 1, 1, 1, 1, 1, 1]
[0, 0, 0, 0, 0, 0, 0, 0, 0]


## Assemble & Run: The Full GA

In [151]:
import random

# ---- Settings ----
CHROM_LENGTH = 9
POP_SIZE = 20
MAX_GENS = 50
MUTATION_RATE = 0.05
CROSSOVER_RATE = 0.8

# ---- Step 1: Create initial population ----
population = []
for i in range(POP_SIZE):
    c = create_chromosome(CHROM_LENGTH)
    population.append(c)

# ---- Step 2: Run the GA ----
for gen in range(MAX_GENS):

    # Calculate fitness of every chromosome
    scores = []
    for c in population:
        scores.append(fitness(c))

    # Find the best chromosome
    best_score = 0
    best_chrom = population[0]
    for i in range(len(population)):
        if scores[i] > best_score:
            best_score = scores[i]
            best_chrom = population[i]

    print("Gen", gen, "| Best fitness:", best_score,
          "| Chromosome:", best_chrom)

    # Check goal
    if best_score == CHROM_LENGTH:
        print("*** Goal reached! ***")
        break

    # ---- Step 3: Build next generation ----
    new_population = []

    while len(new_population) < POP_SIZE:
        p1 = select_parent(population, scores)
        p2 = select_parent(population, scores)

        if random.random() < CROSSOVER_RATE:
            child1, child2 = crossover(p1, p2)
        else:
            child1 = p1[:]
            child2 = p2[:]

        child1 = mutate(child1, MUTATION_RATE)
        child2 = mutate(child2, MUTATION_RATE)

        new_population.append(child1)
        new_population.append(child2)

    population = new_population[:POP_SIZE]

# ---- Final result ----
print()
print("Final population's best chromosome:", best_chrom)
print("Its fitness:", best_score)

Gen 0 | Best fitness: 7 | Chromosome: [1, 1, 1, 1, 1, 0, 1, 1, 0]
Gen 1 | Best fitness: 7 | Chromosome: [1, 1, 1, 1, 1, 0, 1, 1, 0]
Gen 2 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 1, 0, 1, 1, 1]
Gen 3 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 1, 1, 1, 1, 0]
Gen 4 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 0, 1, 1, 1, 1]
Gen 5 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 1, 1, 0, 1, 1]
Gen 6 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 1, 1, 0, 1, 1]
Gen 7 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 1, 1, 0, 1, 1]
Gen 8 | Best fitness: 9 | Chromosome: [1, 1, 1, 1, 1, 1, 1, 1, 1]
*** Goal reached! ***

Final population's best chromosome: [1, 1, 1, 1, 1, 1, 1, 1, 1]
Its fitness: 9


In [152]:
## The algorithm does not always reach the optimal fitness value of 9 because it relies on randomness in selection, 
## crossover, and mutation. In some runs, the population may converge prematurely to suboptimal solutions.

## In our experiments, the algorithm typically reached the optimal solution within 15–25 generations, although this varied between runs.

## Experiment 1: Mutation Rate
### Mutation Rate 0.01

In [153]:
import random

# ---- Settings ----
CHROM_LENGTH = 9
POP_SIZE = 20
MAX_GENS = 50
MUTATION_RATE = 0.01
CROSSOVER_RATE = 0.8

# ---- Step 1: Create initial population ----
population = []
for i in range(POP_SIZE):
    c = create_chromosome(CHROM_LENGTH)
    population.append(c)

# ---- Step 2: Run the GA ----
for gen in range(MAX_GENS):

    # Calculate fitness of every chromosome
    scores = []
    for c in population:
        scores.append(fitness(c))

    # Find the best chromosome
    best_score = 0
    best_chrom = population[0]
    for i in range(len(population)):
        if scores[i] > best_score:
            best_score = scores[i]
            best_chrom = population[i]

    print("Gen", gen, "| Best fitness:", best_score,
          "| Chromosome:", best_chrom)

    # Check goal
    if best_score == CHROM_LENGTH:
        print("*** Goal reached! ***")
        break

    # ---- Step 3: Build next generation ----
    new_population = []

    while len(new_population) < POP_SIZE:
        p1 = select_parent(population, scores)
        p2 = select_parent(population, scores)

        if random.random() < CROSSOVER_RATE:
            child1, child2 = crossover(p1, p2)
        else:
            child1 = p1[:]
            child2 = p2[:]

        child1 = mutate(child1, MUTATION_RATE)
        child2 = mutate(child2, MUTATION_RATE)

        new_population.append(child1)
        new_population.append(child2)

    population = new_population[:POP_SIZE]

# ---- Final result ----
print()
print("Final population's best chromosome:", best_chrom)
print("Its fitness:", best_score)

Gen 0 | Best fitness: 6 | Chromosome: [1, 1, 1, 0, 1, 1, 1, 0, 0]
Gen 1 | Best fitness: 7 | Chromosome: [0, 1, 1, 1, 1, 1, 1, 1, 0]
Gen 2 | Best fitness: 7 | Chromosome: [0, 1, 1, 1, 1, 1, 0, 1, 1]
Gen 3 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 1, 0, 1, 1, 1]
Gen 4 | Best fitness: 7 | Chromosome: [0, 1, 1, 1, 1, 1, 1, 1, 0]
Gen 5 | Best fitness: 8 | Chromosome: [1, 1, 1, 0, 1, 1, 1, 1, 1]
Gen 6 | Best fitness: 8 | Chromosome: [1, 0, 1, 1, 1, 1, 1, 1, 1]
Gen 7 | Best fitness: 9 | Chromosome: [1, 1, 1, 1, 1, 1, 1, 1, 1]
*** Goal reached! ***

Final population's best chromosome: [1, 1, 1, 1, 1, 1, 1, 1, 1]
Its fitness: 9


## Experiment 1: Mutation Rate
### Mutation Rate 0.05

In [154]:
import random

# ---- Settings ----
CHROM_LENGTH = 9
POP_SIZE = 20
MAX_GENS = 50
MUTATION_RATE = 0.05
CROSSOVER_RATE = 0.8

# ---- Step 1: Create initial population ----
population = []
for i in range(POP_SIZE):
    c = create_chromosome(CHROM_LENGTH)
    population.append(c)

# ---- Step 2: Run the GA ----
for gen in range(MAX_GENS):

    # Calculate fitness of every chromosome
    scores = []
    for c in population:
        scores.append(fitness(c))

    # Find the best chromosome
    best_score = 0
    best_chrom = population[0]
    for i in range(len(population)):
        if scores[i] > best_score:
            best_score = scores[i]
            best_chrom = population[i]

    print("Gen", gen, "| Best fitness:", best_score,
          "| Chromosome:", best_chrom)

    # Check goal
    if best_score == CHROM_LENGTH:
        print("*** Goal reached! ***")
        break

    # ---- Step 3: Build next generation ----
    new_population = []

    while len(new_population) < POP_SIZE:
        p1 = select_parent(population, scores)
        p2 = select_parent(population, scores)

        if random.random() < CROSSOVER_RATE:
            child1, child2 = crossover(p1, p2)
        else:
            child1 = p1[:]
            child2 = p2[:]

        child1 = mutate(child1, MUTATION_RATE)
        child2 = mutate(child2, MUTATION_RATE)

        new_population.append(child1)
        new_population.append(child2)

    population = new_population[:POP_SIZE]

# ---- Final result ----
print()
print("Final population's best chromosome:", best_chrom)
print("Its fitness:", best_score)

Gen 0 | Best fitness: 6 | Chromosome: [1, 1, 0, 0, 1, 0, 1, 1, 1]
Gen 1 | Best fitness: 7 | Chromosome: [1, 1, 1, 0, 1, 1, 0, 1, 1]
Gen 2 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 0, 1, 1, 1, 1]
Gen 3 | Best fitness: 7 | Chromosome: [1, 1, 1, 0, 1, 0, 1, 1, 1]
Gen 4 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 0, 1, 1, 1, 1]
Gen 5 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 0, 1, 1, 1, 1]
Gen 6 | Best fitness: 8 | Chromosome: [1, 1, 1, 0, 1, 1, 1, 1, 1]
Gen 7 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 0, 1, 1, 1, 1]
Gen 8 | Best fitness: 8 | Chromosome: [0, 1, 1, 1, 1, 1, 1, 1, 1]
Gen 9 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 0, 1, 1, 1, 1]
Gen 10 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 1, 1, 0, 1, 1]
Gen 11 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 1, 0, 1, 1, 1]
Gen 12 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 0, 1, 1, 1, 1]
Gen 13 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 0, 1, 1, 1, 1]
Gen 14 | Best fitness: 9 | Chromosome: [1, 1, 1, 1, 1, 1, 1, 1, 1]
*** G

## Experiment 1: Mutation Rate
### Mutation Rate 0.2

In [155]:
import random

# ---- Settings ----
CHROM_LENGTH = 9
POP_SIZE = 20
MAX_GENS = 50
MUTATION_RATE = 0.2
CROSSOVER_RATE = 0.8

# ---- Step 1: Create initial population ----
population = []
for i in range(POP_SIZE):
    c = create_chromosome(CHROM_LENGTH)
    population.append(c)

# ---- Step 2: Run the GA ----
for gen in range(MAX_GENS):

    # Calculate fitness of every chromosome
    scores = []
    for c in population:
        scores.append(fitness(c))

    # Find the best chromosome
    best_score = 0
    best_chrom = population[0]
    for i in range(len(population)):
        if scores[i] > best_score:
            best_score = scores[i]
            best_chrom = population[i]

    print("Gen", gen, "| Best fitness:", best_score,
          "| Chromosome:", best_chrom)

    # Check goal
    if best_score == CHROM_LENGTH:
        print("*** Goal reached! ***")
        break

    # ---- Step 3: Build next generation ----
    new_population = []

    while len(new_population) < POP_SIZE:
        p1 = select_parent(population, scores)
        p2 = select_parent(population, scores)

        if random.random() < CROSSOVER_RATE:
            child1, child2 = crossover(p1, p2)
        else:
            child1 = p1[:]
            child2 = p2[:]

        child1 = mutate(child1, MUTATION_RATE)
        child2 = mutate(child2, MUTATION_RATE)

        new_population.append(child1)
        new_population.append(child2)

    population = new_population[:POP_SIZE]

# ---- Final result ----
print()
print("Final population's best chromosome:", best_chrom)
print("Its fitness:", best_score)

Gen 0 | Best fitness: 9 | Chromosome: [1, 1, 1, 1, 1, 1, 1, 1, 1]
*** Goal reached! ***

Final population's best chromosome: [1, 1, 1, 1, 1, 1, 1, 1, 1]
Its fitness: 9


## Experiment 1: Mutation Rate
### Mutation Rate 0.5

In [156]:
import random

# ---- Settings ----
CHROM_LENGTH = 9
POP_SIZE = 20
MAX_GENS = 50
MUTATION_RATE = 0.5
CROSSOVER_RATE = 0.8

# ---- Step 1: Create initial population ----
population = []
for i in range(POP_SIZE):
    c = create_chromosome(CHROM_LENGTH)
    population.append(c)

# ---- Step 2: Run the GA ----
for gen in range(MAX_GENS):

    # Calculate fitness of every chromosome
    scores = []
    for c in population:
        scores.append(fitness(c))

    # Find the best chromosome
    best_score = 0
    best_chrom = population[0]
    for i in range(len(population)):
        if scores[i] > best_score:
            best_score = scores[i]
            best_chrom = population[i]

    print("Gen", gen, "| Best fitness:", best_score,
          "| Chromosome:", best_chrom)

    # Check goal
    if best_score == CHROM_LENGTH:
        print("*** Goal reached! ***")
        break

    # ---- Step 3: Build next generation ----
    new_population = []

    while len(new_population) < POP_SIZE:
        p1 = select_parent(population, scores)
        p2 = select_parent(population, scores)

        if random.random() < CROSSOVER_RATE:
            child1, child2 = crossover(p1, p2)
        else:
            child1 = p1[:]
            child2 = p2[:]

        child1 = mutate(child1, MUTATION_RATE)
        child2 = mutate(child2, MUTATION_RATE)

        new_population.append(child1)
        new_population.append(child2)

    population = new_population[:POP_SIZE]

# ---- Final result ----
print()
print("Final population's best chromosome:", best_chrom)
print("Its fitness:", best_score)

Gen 0 | Best fitness: 6 | Chromosome: [0, 0, 0, 1, 1, 1, 1, 1, 1]
Gen 1 | Best fitness: 9 | Chromosome: [1, 1, 1, 1, 1, 1, 1, 1, 1]
*** Goal reached! ***

Final population's best chromosome: [1, 1, 1, 1, 1, 1, 1, 1, 1]
Its fitness: 9


## Experiment 2: Population Size
### Pop size 4

In [157]:
import random

# ---- Settings ----
CHROM_LENGTH = 9
POP_SIZE = 4
MAX_GENS = 50
MUTATION_RATE = 0.05
CROSSOVER_RATE = 0.8

# ---- Step 1: Create initial population ----
population = []
for i in range(POP_SIZE):
    c = create_chromosome(CHROM_LENGTH)
    population.append(c)

# ---- Step 2: Run the GA ----
for gen in range(MAX_GENS):

    # Calculate fitness of every chromosome
    scores = []
    for c in population:
        scores.append(fitness(c))

    # Find the best chromosome
    best_score = 0
    best_chrom = population[0]
    for i in range(len(population)):
        if scores[i] > best_score:
            best_score = scores[i]
            best_chrom = population[i]

    print("Gen", gen, "| Best fitness:", best_score,
          "| Chromosome:", best_chrom)

    # Check goal
    if best_score == CHROM_LENGTH:
        print("*** Goal reached! ***")
        break

    # ---- Step 3: Build next generation ----
    new_population = []

    while len(new_population) < POP_SIZE:
        p1 = select_parent(population, scores)
        p2 = select_parent(population, scores)

        if random.random() < CROSSOVER_RATE:
            child1, child2 = crossover(p1, p2)
        else:
            child1 = p1[:]
            child2 = p2[:]

        child1 = mutate(child1, MUTATION_RATE)
        child2 = mutate(child2, MUTATION_RATE)

        new_population.append(child1)
        new_population.append(child2)

    population = new_population[:POP_SIZE]

# ---- Final result ----
print()
print("Final population's best chromosome:", best_chrom)
print("Its fitness:", best_score)

Gen 0 | Best fitness: 7 | Chromosome: [1, 1, 1, 1, 1, 1, 1, 0, 0]
Gen 1 | Best fitness: 6 | Chromosome: [0, 1, 1, 1, 1, 1, 1, 0, 0]
Gen 2 | Best fitness: 6 | Chromosome: [0, 1, 1, 1, 1, 1, 1, 0, 0]
Gen 3 | Best fitness: 5 | Chromosome: [0, 1, 1, 1, 0, 0, 1, 1, 0]
Gen 4 | Best fitness: 5 | Chromosome: [1, 1, 1, 0, 0, 0, 1, 1, 0]
Gen 5 | Best fitness: 5 | Chromosome: [1, 1, 1, 0, 0, 0, 1, 1, 0]
Gen 6 | Best fitness: 7 | Chromosome: [1, 1, 1, 1, 0, 1, 1, 1, 0]
Gen 7 | Best fitness: 7 | Chromosome: [1, 1, 1, 1, 0, 1, 1, 1, 0]
Gen 8 | Best fitness: 7 | Chromosome: [1, 1, 1, 1, 0, 1, 1, 1, 0]
Gen 9 | Best fitness: 7 | Chromosome: [1, 1, 1, 1, 0, 1, 1, 1, 0]
Gen 10 | Best fitness: 6 | Chromosome: [1, 1, 1, 1, 0, 0, 1, 1, 0]
Gen 11 | Best fitness: 6 | Chromosome: [1, 1, 1, 1, 0, 0, 1, 1, 0]
Gen 12 | Best fitness: 7 | Chromosome: [1, 1, 1, 1, 1, 0, 1, 1, 0]
Gen 13 | Best fitness: 7 | Chromosome: [1, 1, 1, 1, 1, 1, 0, 1, 0]
Gen 14 | Best fitness: 7 | Chromosome: [1, 1, 1, 1, 1, 1, 0, 1, 0]
Gen 1

## Experiment 2: Population Size
### Pop size 10

In [158]:
import random

# ---- Settings ----
CHROM_LENGTH = 9
POP_SIZE = 10
MAX_GENS = 50
MUTATION_RATE = 0.05
CROSSOVER_RATE = 0.8

# ---- Step 1: Create initial population ----
population = []
for i in range(POP_SIZE):
    c = create_chromosome(CHROM_LENGTH)
    population.append(c)

# ---- Step 2: Run the GA ----
for gen in range(MAX_GENS):

    # Calculate fitness of every chromosome
    scores = []
    for c in population:
        scores.append(fitness(c))

    # Find the best chromosome
    best_score = 0
    best_chrom = population[0]
    for i in range(len(population)):
        if scores[i] > best_score:
            best_score = scores[i]
            best_chrom = population[i]

    print("Gen", gen, "| Best fitness:", best_score,
          "| Chromosome:", best_chrom)

    # Check goal
    if best_score == CHROM_LENGTH:
        print("*** Goal reached! ***")
        break

    # ---- Step 3: Build next generation ----
    new_population = []

    while len(new_population) < POP_SIZE:
        p1 = select_parent(population, scores)
        p2 = select_parent(population, scores)

        if random.random() < CROSSOVER_RATE:
            child1, child2 = crossover(p1, p2)
        else:
            child1 = p1[:]
            child2 = p2[:]

        child1 = mutate(child1, MUTATION_RATE)
        child2 = mutate(child2, MUTATION_RATE)

        new_population.append(child1)
        new_population.append(child2)

    population = new_population[:POP_SIZE]

# ---- Final result ----
print()
print("Final population's best chromosome:", best_chrom)
print("Its fitness:", best_score)

Gen 0 | Best fitness: 6 | Chromosome: [1, 1, 1, 1, 0, 1, 0, 0, 1]
Gen 1 | Best fitness: 7 | Chromosome: [0, 1, 1, 1, 1, 0, 1, 1, 1]
Gen 2 | Best fitness: 7 | Chromosome: [0, 1, 1, 1, 0, 1, 1, 1, 1]
Gen 3 | Best fitness: 7 | Chromosome: [1, 0, 1, 1, 1, 0, 1, 1, 1]
Gen 4 | Best fitness: 7 | Chromosome: [0, 1, 1, 1, 1, 0, 1, 1, 1]
Gen 5 | Best fitness: 7 | Chromosome: [0, 1, 1, 1, 1, 0, 1, 1, 1]
Gen 6 | Best fitness: 8 | Chromosome: [1, 0, 1, 1, 1, 1, 1, 1, 1]
Gen 7 | Best fitness: 7 | Chromosome: [1, 0, 1, 1, 1, 0, 1, 1, 1]
Gen 8 | Best fitness: 8 | Chromosome: [1, 1, 1, 0, 1, 1, 1, 1, 1]
Gen 9 | Best fitness: 9 | Chromosome: [1, 1, 1, 1, 1, 1, 1, 1, 1]
*** Goal reached! ***

Final population's best chromosome: [1, 1, 1, 1, 1, 1, 1, 1, 1]
Its fitness: 9


## Experiment 2: Population Size
### Pop size 20

In [159]:
import random

# ---- Settings ----
CHROM_LENGTH = 9
POP_SIZE = 20
MAX_GENS = 50
MUTATION_RATE = 0.05
CROSSOVER_RATE = 0.8

# ---- Step 1: Create initial population ----
population = []
for i in range(POP_SIZE):
    c = create_chromosome(CHROM_LENGTH)
    population.append(c)

# ---- Step 2: Run the GA ----
for gen in range(MAX_GENS):

    # Calculate fitness of every chromosome
    scores = []
    for c in population:
        scores.append(fitness(c))

    # Find the best chromosome
    best_score = 0
    best_chrom = population[0]
    for i in range(len(population)):
        if scores[i] > best_score:
            best_score = scores[i]
            best_chrom = population[i]

    print("Gen", gen, "| Best fitness:", best_score,
          "| Chromosome:", best_chrom)

    # Check goal
    if best_score == CHROM_LENGTH:
        print("*** Goal reached! ***")
        break

    # ---- Step 3: Build next generation ----
    new_population = []

    while len(new_population) < POP_SIZE:
        p1 = select_parent(population, scores)
        p2 = select_parent(population, scores)

        if random.random() < CROSSOVER_RATE:
            child1, child2 = crossover(p1, p2)
        else:
            child1 = p1[:]
            child2 = p2[:]

        child1 = mutate(child1, MUTATION_RATE)
        child2 = mutate(child2, MUTATION_RATE)

        new_population.append(child1)
        new_population.append(child2)

    population = new_population[:POP_SIZE]

# ---- Final result ----
print()
print("Final population's best chromosome:", best_chrom)
print("Its fitness:", best_score)

Gen 0 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 1, 1, 1, 0, 1]
Gen 1 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 1, 1, 1, 0, 1]
Gen 2 | Best fitness: 8 | Chromosome: [1, 1, 0, 1, 1, 1, 1, 1, 1]
Gen 3 | Best fitness: 8 | Chromosome: [1, 0, 1, 1, 1, 1, 1, 1, 1]
Gen 4 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 0, 1, 1, 1, 1]
Gen 5 | Best fitness: 8 | Chromosome: [1, 0, 1, 1, 1, 1, 1, 1, 1]
Gen 6 | Best fitness: 9 | Chromosome: [1, 1, 1, 1, 1, 1, 1, 1, 1]
*** Goal reached! ***

Final population's best chromosome: [1, 1, 1, 1, 1, 1, 1, 1, 1]
Its fitness: 9


## Experiment 2: Population Size
### Pop size 50

In [160]:
import random

# ---- Settings ----
CHROM_LENGTH = 9
POP_SIZE = 50
MAX_GENS = 50
MUTATION_RATE = 0.05
CROSSOVER_RATE = 0.8

# ---- Step 1: Create initial population ----
population = []
for i in range(POP_SIZE):
    c = create_chromosome(CHROM_LENGTH)
    population.append(c)

# ---- Step 2: Run the GA ----
for gen in range(MAX_GENS):

    # Calculate fitness of every chromosome
    scores = []
    for c in population:
        scores.append(fitness(c))

    # Find the best chromosome
    best_score = 0
    best_chrom = population[0]
    for i in range(len(population)):
        if scores[i] > best_score:
            best_score = scores[i]
            best_chrom = population[i]

    print("Gen", gen, "| Best fitness:", best_score,
          "| Chromosome:", best_chrom)

    # Check goal
    if best_score == CHROM_LENGTH:
        print("*** Goal reached! ***")
        break

    # ---- Step 3: Build next generation ----
    new_population = []

    while len(new_population) < POP_SIZE:
        p1 = select_parent(population, scores)
        p2 = select_parent(population, scores)

        if random.random() < CROSSOVER_RATE:
            child1, child2 = crossover(p1, p2)
        else:
            child1 = p1[:]
            child2 = p2[:]

        child1 = mutate(child1, MUTATION_RATE)
        child2 = mutate(child2, MUTATION_RATE)

        new_population.append(child1)
        new_population.append(child2)

    population = new_population[:POP_SIZE]

# ---- Final result ----
print()
print("Final population's best chromosome:", best_chrom)
print("Its fitness:", best_score)

Gen 0 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 1, 0, 1, 1, 1]
Gen 1 | Best fitness: 8 | Chromosome: [1, 0, 1, 1, 1, 1, 1, 1, 1]
Gen 2 | Best fitness: 8 | Chromosome: [1, 0, 1, 1, 1, 1, 1, 1, 1]
Gen 3 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 1, 0, 1, 1, 1]
Gen 4 | Best fitness: 9 | Chromosome: [1, 1, 1, 1, 1, 1, 1, 1, 1]
*** Goal reached! ***

Final population's best chromosome: [1, 1, 1, 1, 1, 1, 1, 1, 1]
Its fitness: 9


## Experiment 3: Crossover Rate
### Crossover Rate 0.0

In [161]:
import random

# ---- Settings ----
CHROM_LENGTH = 9
POP_SIZE = 20
MAX_GENS = 50
MUTATION_RATE = 0.05
CROSSOVER_RATE = 0.0

# ---- Step 1: Create initial population ----
population = []
for i in range(POP_SIZE):
    c = create_chromosome(CHROM_LENGTH)
    population.append(c)

# ---- Step 2: Run the GA ----
for gen in range(MAX_GENS):

    # Calculate fitness of every chromosome
    scores = []
    for c in population:
        scores.append(fitness(c))

    # Find the best chromosome
    best_score = 0
    best_chrom = population[0]
    for i in range(len(population)):
        if scores[i] > best_score:
            best_score = scores[i]
            best_chrom = population[i]

    print("Gen", gen, "| Best fitness:", best_score,
          "| Chromosome:", best_chrom)

    # Check goal
    if best_score == CHROM_LENGTH:
        print("*** Goal reached! ***")
        break

    # ---- Step 3: Build next generation ----
    new_population = []

    while len(new_population) < POP_SIZE:
        p1 = select_parent(population, scores)
        p2 = select_parent(population, scores)

        if random.random() < CROSSOVER_RATE:
            child1, child2 = crossover(p1, p2)
        else:
            child1 = p1[:]
            child2 = p2[:]

        child1 = mutate(child1, MUTATION_RATE)
        child2 = mutate(child2, MUTATION_RATE)

        new_population.append(child1)
        new_population.append(child2)

    population = new_population[:POP_SIZE]


# ---- Final result ----
print()
print("Final population's best chromosome:", best_chrom)
print("Its fitness:", best_score)

Gen 0 | Best fitness: 6 | Chromosome: [0, 1, 1, 0, 1, 1, 1, 1, 0]
Gen 1 | Best fitness: 7 | Chromosome: [1, 1, 1, 1, 1, 0, 1, 1, 0]
Gen 2 | Best fitness: 7 | Chromosome: [1, 1, 1, 0, 1, 1, 1, 1, 0]
Gen 3 | Best fitness: 7 | Chromosome: [0, 1, 1, 1, 0, 1, 1, 1, 1]
Gen 4 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 1, 1, 1, 1, 0]
Gen 5 | Best fitness: 8 | Chromosome: [0, 1, 1, 1, 1, 1, 1, 1, 1]
Gen 6 | Best fitness: 7 | Chromosome: [0, 1, 1, 1, 1, 1, 1, 0, 1]
Gen 7 | Best fitness: 7 | Chromosome: [0, 1, 1, 1, 1, 1, 1, 0, 1]
Gen 8 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 0, 1, 1, 1, 1]
Gen 9 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 0, 1, 1, 1, 1]
Gen 10 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 0, 1, 1, 1, 1]
Gen 11 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 0, 1, 1, 1, 1]
Gen 12 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 0, 1, 1, 1, 1]
Gen 13 | Best fitness: 8 | Chromosome: [1, 1, 1, 1, 0, 1, 1, 1, 1]
Gen 14 | Best fitness: 7 | Chromosome: [1, 1, 1, 1, 1, 1, 0, 1, 0]
Gen 1

## Experiment 3: Crossover Rate
### Crossover Rate 0.5

In [162]:
import random

# ---- Settings ----
CHROM_LENGTH = 9
POP_SIZE = 20
MAX_GENS = 50
MUTATION_RATE = 0.05
CROSSOVER_RATE = 0.5

# ---- Step 1: Create initial population ----
population = []
for i in range(POP_SIZE):
    c = create_chromosome(CHROM_LENGTH)
    population.append(c)

# ---- Step 2: Run the GA ----
for gen in range(MAX_GENS):

    # Calculate fitness of every chromosome
    scores = []
    for c in population:
        scores.append(fitness(c))

    # Find the best chromosome
    best_score = 0
    best_chrom = population[0]
    for i in range(len(population)):
        if scores[i] > best_score:
            best_score = scores[i]
            best_chrom = population[i]

    print("Gen", gen, "| Best fitness:", best_score,
          "| Chromosome:", best_chrom)

    # Check goal
    if best_score == CHROM_LENGTH:
        print("*** Goal reached! ***")
        break

    # ---- Step 3: Build next generation ----
    new_population = []

    while len(new_population) < POP_SIZE:
        p1 = select_parent(population, scores)
        p2 = select_parent(population, scores)

        if random.random() < CROSSOVER_RATE:
            child1, child2 = crossover(p1, p2)
        else:
            child1 = p1[:]
            child2 = p2[:]

        child1 = mutate(child1, MUTATION_RATE)
        child2 = mutate(child2, MUTATION_RATE)

        new_population.append(child1)
        new_population.append(child2)

    population = new_population[:POP_SIZE]

# ---- Final result ----
print()
print("Final population's best chromosome:", best_chrom)
print("Its fitness:", best_score)

Gen 0 | Best fitness: 6 | Chromosome: [1, 1, 1, 1, 1, 0, 0, 1, 0]
Gen 1 | Best fitness: 7 | Chromosome: [1, 1, 1, 1, 0, 0, 1, 1, 1]
Gen 2 | Best fitness: 7 | Chromosome: [1, 1, 1, 1, 0, 0, 1, 1, 1]
Gen 3 | Best fitness: 6 | Chromosome: [1, 1, 0, 0, 1, 1, 0, 1, 1]
Gen 4 | Best fitness: 6 | Chromosome: [1, 1, 0, 1, 1, 0, 0, 1, 1]
Gen 5 | Best fitness: 6 | Chromosome: [1, 1, 0, 0, 1, 1, 1, 0, 1]
Gen 6 | Best fitness: 7 | Chromosome: [0, 1, 1, 1, 1, 1, 0, 1, 1]
Gen 7 | Best fitness: 6 | Chromosome: [0, 0, 1, 1, 1, 0, 1, 1, 1]
Gen 8 | Best fitness: 7 | Chromosome: [1, 1, 1, 1, 0, 0, 1, 1, 1]
Gen 9 | Best fitness: 6 | Chromosome: [1, 1, 0, 0, 0, 1, 1, 1, 1]
Gen 10 | Best fitness: 7 | Chromosome: [1, 1, 0, 0, 1, 1, 1, 1, 1]
Gen 11 | Best fitness: 7 | Chromosome: [1, 1, 0, 0, 1, 1, 1, 1, 1]
Gen 12 | Best fitness: 8 | Chromosome: [1, 0, 1, 1, 1, 1, 1, 1, 1]
Gen 13 | Best fitness: 6 | Chromosome: [1, 1, 0, 1, 0, 0, 1, 1, 1]
Gen 14 | Best fitness: 6 | Chromosome: [1, 1, 1, 1, 0, 0, 1, 0, 1]
Gen 1

## Experiment 3: Crossover Rate
### Crossover Rate 0.8

In [163]:
import random

# ---- Settings ----
CHROM_LENGTH = 9
POP_SIZE = 20
MAX_GENS = 50
MUTATION_RATE = 0.05
CROSSOVER_RATE = 0.8

# ---- Step 1: Create initial population ----
population = []
for i in range(POP_SIZE):
    c = create_chromosome(CHROM_LENGTH)
    population.append(c)

# ---- Step 2: Run the GA ----
for gen in range(MAX_GENS):

    # Calculate fitness of every chromosome
    scores = []
    for c in population:
        scores.append(fitness(c))

    # Find the best chromosome
    best_score = 0
    best_chrom = population[0]
    for i in range(len(population)):
        if scores[i] > best_score:
            best_score = scores[i]
            best_chrom = population[i]

    print("Gen", gen, "| Best fitness:", best_score,
          "| Chromosome:", best_chrom)

    # Check goal
    if best_score == CHROM_LENGTH:
        print("*** Goal reached! ***")
        break

    # ---- Step 3: Build next generation ----
    new_population = []

    while len(new_population) < POP_SIZE:
        p1 = select_parent(population, scores)
        p2 = select_parent(population, scores)

        if random.random() < CROSSOVER_RATE:
            child1, child2 = crossover(p1, p2)
        else:
            child1 = p1[:]
            child2 = p2[:]

        child1 = mutate(child1, MUTATION_RATE)
        child2 = mutate(child2, MUTATION_RATE)

        new_population.append(child1)
        new_population.append(child2)

    population = new_population[:POP_SIZE]

# ---- Final result ----
print()
print("Final population's best chromosome:", best_chrom)
print("Its fitness:", best_score)

Gen 0 | Best fitness: 7 | Chromosome: [1, 1, 1, 1, 1, 0, 0, 1, 1]
Gen 1 | Best fitness: 7 | Chromosome: [0, 1, 0, 1, 1, 1, 1, 1, 1]
Gen 2 | Best fitness: 6 | Chromosome: [0, 1, 1, 1, 0, 1, 1, 1, 0]
Gen 3 | Best fitness: 6 | Chromosome: [0, 0, 1, 1, 0, 1, 1, 1, 1]
Gen 4 | Best fitness: 7 | Chromosome: [1, 0, 1, 1, 0, 1, 1, 1, 1]
Gen 5 | Best fitness: 6 | Chromosome: [0, 0, 1, 1, 1, 0, 1, 1, 1]
Gen 6 | Best fitness: 6 | Chromosome: [1, 0, 1, 1, 0, 1, 1, 0, 1]
Gen 7 | Best fitness: 6 | Chromosome: [0, 1, 1, 1, 1, 0, 1, 1, 0]
Gen 8 | Best fitness: 6 | Chromosome: [0, 1, 1, 1, 0, 1, 1, 1, 0]
Gen 9 | Best fitness: 8 | Chromosome: [0, 1, 1, 1, 1, 1, 1, 1, 1]
Gen 10 | Best fitness: 9 | Chromosome: [1, 1, 1, 1, 1, 1, 1, 1, 1]
*** Goal reached! ***

Final population's best chromosome: [1, 1, 1, 1, 1, 1, 1, 1, 1]
Its fitness: 9


## Experiment 3: Crossover Rate
### Crossover Rate 1.0

In [164]:
import random

# ---- Settings ----
CHROM_LENGTH = 9
POP_SIZE = 20
MAX_GENS = 50
MUTATION_RATE = 0.05
CROSSOVER_RATE = 1.0

# ---- Step 1: Create initial population ----
population = []
for i in range(POP_SIZE):
    c = create_chromosome(CHROM_LENGTH)
    population.append(c)

# ---- Step 2: Run the GA ----
for gen in range(MAX_GENS):

    # Calculate fitness of every chromosome
    scores = []
    for c in population:
        scores.append(fitness(c))

    # Find the best chromosome
    best_score = 0
    best_chrom = population[0]
    for i in range(len(population)):
        if scores[i] > best_score:
            best_score = scores[i]
            best_chrom = population[i]

    print("Gen", gen, "| Best fitness:", best_score,
          "| Chromosome:", best_chrom)

    # Check goal
    if best_score == CHROM_LENGTH:
        print("*** Goal reached! ***")
        break

    # ---- Step 3: Build next generation ----
    new_population = []

    while len(new_population) < POP_SIZE:
        p1 = select_parent(population, scores)
        p2 = select_parent(population, scores)

        if random.random() < CROSSOVER_RATE:
            child1, child2 = crossover(p1, p2)
        else:
            child1 = p1[:]
            child2 = p2[:]

        child1 = mutate(child1, MUTATION_RATE)
        child2 = mutate(child2, MUTATION_RATE)

        new_population.append(child1)
        new_population.append(child2)

    population = new_population[:POP_SIZE]

# ---- Final result ----
print()
print("Final population's best chromosome:", best_chrom)
print("Its fitness:", best_score)

Gen 0 | Best fitness: 7 | Chromosome: [1, 0, 1, 1, 1, 1, 1, 1, 0]
Gen 1 | Best fitness: 9 | Chromosome: [1, 1, 1, 1, 1, 1, 1, 1, 1]
*** Goal reached! ***

Final population's best chromosome: [1, 1, 1, 1, 1, 1, 1, 1, 1]
Its fitness: 9


# Challenge: Target String Matching

In [165]:
import random
import string

TARGET = "HELLO"
LETTERS = string.ascii_uppercase + " "
POP_SIZE = 100
MUTATION_RATE = 0.05
CROSSOVER_RATE = 0.8
MAX_GENS = 500

def create_random_string(length):
    return [random.choice(LETTERS) for _ in range(length)]

def fitness(chromosome):
    count = 0
    for i in range(len(chromosome)):
        if chromosome[i] == TARGET[i]:
            count += 1
    return count

def select_parent(population, scores):
    total = sum(scores)
    spin = random.uniform(0, total)
    
    running_sum = 0
    for i in range(len(population)):
        running_sum += scores[i]
        if spin <= running_sum:
            return population[i]
    
    return population[-1]

def crossover(p1, p2):
    point = random.randint(1, len(p1) - 1)
    c1 = p1[:point] + p2[point:]
    c2 = p2[:point] + p1[point:]
    return c1, c2

def mutate(chromosome, rate):
    new_chrom = []
    for gene in chromosome:
        if random.random() < rate:
            new_chrom.append(random.choice(LETTERS))
        else:
            new_chrom.append(gene)
    return new_chrom

# ---- Main GA Loop ----
population = [create_random_string(len(TARGET)) for _ in range(POP_SIZE)]

for gen in range(MAX_GENS):

    scores = [fitness(c) for c in population]

    best_score = max(scores)
    best_chrom = population[scores.index(best_score)]

    best_string = "".join(best_chrom)

    print("Gen", gen, "| Best:", best_string,
          "| Fitness:", best_score, "/", len(TARGET))

    if best_score == len(TARGET):
        print("*** Target matched! ***")
        break

    new_population = []
    while len(new_population) < POP_SIZE:
        p1 = select_parent(population, scores)
        p2 = select_parent(population, scores)

        if random.random() < CROSSOVER_RATE:
            c1, c2 = crossover(p1, p2)
        else:
            c1, c2 = p1[:], p2[:]

        new_population.append(mutate(c1, MUTATION_RATE))
        new_population.append(mutate(c2, MUTATION_RATE))

    population = new_population[:POP_SIZE]

Gen 0 | Best: HESIU | Fitness: 2 / 5
Gen 1 | Best: HESLE | Fitness: 3 / 5
Gen 2 | Best: HOLQO | Fitness: 3 / 5
Gen 3 | Best: HOLLY | Fitness: 3 / 5
Gen 4 | Best: HOLLO | Fitness: 4 / 5
Gen 5 | Best: HOLLO | Fitness: 4 / 5
Gen 6 | Best: HELLO | Fitness: 5 / 5
*** Target matched! ***


In [166]:
import random
import string

TARGET = "ABBAS AHMED"
LETTERS = string.ascii_uppercase + " "
POP_SIZE = 100
MUTATION_RATE = 0.05
CROSSOVER_RATE = 0.8
MAX_GENS = 500

def create_random_string(length):
    return [random.choice(LETTERS) for _ in range(length)]

def fitness(chromosome):
    count = 0
    for i in range(len(chromosome)):
        if chromosome[i] == TARGET[i]:
            count += 1
    return count

def select_parent(population, scores):
    total = sum(scores)
    spin = random.uniform(0, total)
    
    running_sum = 0
    for i in range(len(population)):
        running_sum += scores[i]
        if spin <= running_sum:
            return population[i]
    
    return population[-1]

def crossover(p1, p2):
    point = random.randint(1, len(p1) - 1)
    c1 = p1[:point] + p2[point:]
    c2 = p2[:point] + p1[point:]
    return c1, c2

def mutate(chromosome, rate):
    new_chrom = []
    for gene in chromosome:
        if random.random() < rate:
            new_chrom.append(random.choice(LETTERS))
        else:
            new_chrom.append(gene)
    return new_chrom

# ---- Main GA Loop ----
population = [create_random_string(len(TARGET)) for _ in range(POP_SIZE)]

for gen in range(MAX_GENS):

    scores = [fitness(c) for c in population]

    best_score = max(scores)
    best_chrom = population[scores.index(best_score)]

    best_string = "".join(best_chrom)

    print("Gen", gen, "| Best:", best_string,
          "| Fitness:", best_score, "/", len(TARGET))

    if best_score == len(TARGET):
        print("*** Target matched! ***")
        break

    new_population = []
    while len(new_population) < POP_SIZE:
        p1 = select_parent(population, scores)
        p2 = select_parent(population, scores)

        if random.random() < CROSSOVER_RATE:
            c1, c2 = crossover(p1, p2)
        else:
            c1, c2 = p1[:], p2[:]

        new_population.append(mutate(c1, MUTATION_RATE))
        new_population.append(mutate(c2, MUTATION_RATE))

    population = new_population[:POP_SIZE]

Gen 0 | Best: AZRVS V NIC | Fitness: 3 / 11
Gen 1 | Best: AZRVS VQI D | Fitness: 4 / 11
Gen 2 | Best: AZBVS VQI D | Fitness: 5 / 11
Gen 3 | Best: ABRVS VQI D | Fitness: 5 / 11
Gen 4 | Best: XBQGS BRDED | Fitness: 5 / 11
Gen 5 | Best: ABBAUFTHTED | Fitness: 7 / 11
Gen 6 | Best: ABBAUFTHTED | Fitness: 7 / 11
Gen 7 | Best: CBQAM AHIHD | Fitness: 6 / 11
Gen 8 | Best: CBQAY KHIED | Fitness: 6 / 11
Gen 9 | Best: IBQAP AHSMD | Fitness: 6 / 11
Gen 10 | Best: YBBAE VHPED | Fitness: 7 / 11
Gen 11 | Best: ABBVS VHZ D | Fitness: 7 / 11
Gen 12 | Best: ABBAUFAACED | Fitness: 7 / 11
Gen 13 | Best: ABBAS AHHMD | Fitness: 9 / 11
Gen 14 | Best: ABBAS AHHMD | Fitness: 9 / 11
Gen 15 | Best: ABBAS AHHMD | Fitness: 9 / 11
Gen 16 | Best: ARBAS AHCED | Fitness: 9 / 11
Gen 17 | Best: ARBAS AHYED | Fitness: 9 / 11
Gen 18 | Best: ABAAM AHHED | Fitness: 8 / 11
Gen 19 | Best: ASBAS AHYED | Fitness: 9 / 11
Gen 20 | Best: YBBAS AHYED | Fitness: 9 / 11
Gen 21 | Best: YBBAS AHCED | Fitness: 9 / 11
Gen 22 | Best: ABBAS

In [167]:
import random
import string

TARGET = "INTELLIGENCE"
LETTERS = string.ascii_uppercase + " "
POP_SIZE = 100
MUTATION_RATE = 0.05
CROSSOVER_RATE = 0.8
MAX_GENS = 500

def create_random_string(length):
    return [random.choice(LETTERS) for _ in range(length)]

def fitness(chromosome):
    count = 0
    for i in range(len(chromosome)):
        if chromosome[i] == TARGET[i]:
            count += 1
    return count

def select_parent(population, scores):
    total = sum(scores)
    spin = random.uniform(0, total)
    
    running_sum = 0
    for i in range(len(population)):
        running_sum += scores[i]
        if spin <= running_sum:
            return population[i]
    
    return population[-1]

def crossover(p1, p2):
    point = random.randint(1, len(p1) - 1)
    c1 = p1[:point] + p2[point:]
    c2 = p2[:point] + p1[point:]
    return c1, c2

def mutate(chromosome, rate):
    new_chrom = []
    for gene in chromosome:
        if random.random() < rate:
            new_chrom.append(random.choice(LETTERS))
        else:
            new_chrom.append(gene)
    return new_chrom

# ---- Main GA Loop ----
population = [create_random_string(len(TARGET)) for _ in range(POP_SIZE)]

for gen in range(MAX_GENS):

    scores = [fitness(c) for c in population]

    best_score = max(scores)
    best_chrom = population[scores.index(best_score)]

    best_string = "".join(best_chrom)

    print("Gen", gen, "| Best:", best_string,
          "| Fitness:", best_score, "/", len(TARGET))

    if best_score == len(TARGET):
        print("*** Target matched! ***")
        break

    new_population = []
    while len(new_population) < POP_SIZE:
        p1 = select_parent(population, scores)
        p2 = select_parent(population, scores)

        if random.random() < CROSSOVER_RATE:
            c1, c2 = crossover(p1, p2)
        else:
            c1, c2 = p1[:], p2[:]

        new_population.append(mutate(c1, MUTATION_RATE))
        new_population.append(mutate(c2, MUTATION_RATE))

    population = new_population[:POP_SIZE]

Gen 0 | Best: INJJDKFNMFAE | Fitness: 3 / 12
Gen 1 | Best: INJSANIXXZCD | Fitness: 4 / 12
Gen 2 | Best: INJJANIXXZCD | Fitness: 4 / 12
Gen 3 | Best: INUEQNIXXZCD | Fitness: 5 / 12
Gen 4 | Best: INUEQNIXXZAE | Fitness: 5 / 12
Gen 5 | Best: INUEQPIXXZCJ | Fitness: 5 / 12
Gen 6 | Best: INUEQKIJXKBE | Fitness: 5 / 12
Gen 7 | Best: INEPLSIVXZCE | Fitness: 6 / 12
Gen 8 | Best: INJJDPIXXZCE | Fitness: 5 / 12
Gen 9 | Best: INEPLSITDNCK | Fitness: 6 / 12
Gen 10 | Best: IN ERBYNENTE | Fitness: 6 / 12
Gen 11 | Best: INEPDSINENTE | Fitness: 6 / 12
Gen 12 | Best: IN EFXIGENBJ | Fitness: 7 / 12
Gen 13 | Best: INUEBCYGEWTE | Fitness: 6 / 12
Gen 14 | Best: INUENBIXESCE | Fitness: 7 / 12
Gen 15 | Best: INUENBIXESCE | Fitness: 7 / 12
Gen 16 | Best: IXTEAQIXXNCE | Fitness: 7 / 12
Gen 17 | Best: INTERCIHXNHE | Fitness: 7 / 12
Gen 18 | Best: INUERCIHXNNE | Fitness: 6 / 12
Gen 19 | Best: INNELWIXXNCE | Fitness: 8 / 12
Gen 20 | Best: IFHPLEIXENCE | Fitness: 7 / 12
Gen 21 | Best: INTEULIGXNCE | Fitness: 10 / 

In [168]:
import random
import string

TARGET = "GENETIC"
LETTERS = string.ascii_uppercase + " "
POP_SIZE = 100
MUTATION_RATE = 0.05
CROSSOVER_RATE = 0.8
MAX_GENS = 500

def create_random_string(length):
    return [random.choice(LETTERS) for _ in range(length)]

def fitness(chromosome):
    count = 0
    for i in range(len(chromosome)):
        if chromosome[i] == TARGET[i]:
            count += 1
    return count

def select_parent(population, scores):
    total = sum(scores)
    spin = random.uniform(0, total)
    
    running_sum = 0
    for i in range(len(population)):
        running_sum += scores[i]
        if spin <= running_sum:
            return population[i]
    
    return population[-1]

def crossover(p1, p2):
    point = random.randint(1, len(p1) - 1)
    c1 = p1[:point] + p2[point:]
    c2 = p2[:point] + p1[point:]
    return c1, c2

def mutate(chromosome, rate):
    new_chrom = []
    for gene in chromosome:
        if random.random() < rate:
            new_chrom.append(random.choice(LETTERS))
        else:
            new_chrom.append(gene)
    return new_chrom

# ---- Main GA Loop ----
population = [create_random_string(len(TARGET)) for _ in range(POP_SIZE)]

for gen in range(MAX_GENS):

    scores = [fitness(c) for c in population]

    best_score = max(scores)
    best_chrom = population[scores.index(best_score)]

    best_string = "".join(best_chrom)

    print("Gen", gen, "| Best:", best_string,
          "| Fitness:", best_score, "/", len(TARGET))

    if best_score == len(TARGET):
        print("*** Target matched! ***")
        break

    new_population = []
    while len(new_population) < POP_SIZE:
        p1 = select_parent(population, scores)
        p2 = select_parent(population, scores)

        if random.random() < CROSSOVER_RATE:
            c1, c2 = crossover(p1, p2)
        else:
            c1, c2 = p1[:], p2[:]

        new_population.append(mutate(c1, MUTATION_RATE))
        new_population.append(mutate(c2, MUTATION_RATE))

    population = new_population[:POP_SIZE]

Gen 0 | Best: PKHEXIA | Fitness: 2 / 7
Gen 1 | Best: ZECATIE | Fitness: 3 / 7
Gen 2 | Best: ZECEXIB | Fitness: 3 / 7
Gen 3 | Best: VEDECIO | Fitness: 3 / 7
Gen 4 | Best: EEAOTIC | Fitness: 4 / 7
Gen 5 | Best: MEIETIE | Fitness: 4 / 7
Gen 6 | Best: MEIETIE | Fitness: 4 / 7
Gen 7 | Best: MEIETIE | Fitness: 4 / 7
Gen 8 | Best: GWNETIZ | Fitness: 5 / 7
Gen 9 | Best: ZENEYIC | Fitness: 5 / 7
Gen 10 | Best: KENEOIC | Fitness: 5 / 7
Gen 11 | Best: GNNETIC | Fitness: 6 / 7
Gen 12 | Best: GNNETIC | Fitness: 6 / 7
Gen 13 | Best: GNNETIC | Fitness: 6 / 7
Gen 14 | Best: KENETIC | Fitness: 6 / 7
Gen 15 | Best: GENETIC | Fitness: 7 / 7
*** Target matched! ***


In [184]:
import random
import string

TARGET = "GEnetic"
LETTERS = string.ascii_uppercase + string.ascii_lowercase + " "
POP_SIZE = 100
MUTATION_RATE = 0.05
CROSSOVER_RATE = 0.8
MAX_GENS = 500

def create_random_string(length):
    return [random.choice(LETTERS) for _ in range(length)]

def fitness(chromosome):
    count = 0
    for i in range(len(chromosome)):
        if chromosome[i] == TARGET[i]:
            count += 1
    return count

def select_parent(population, scores):
    total = sum(scores)
    spin = random.uniform(0, total)
    
    running_sum = 0
    for i in range(len(population)):
        running_sum += scores[i]
        if spin <= running_sum:
            return population[i]
    
    return population[-1]

def crossover(p1, p2):
    point = random.randint(1, len(p1) - 1)
    c1 = p1[:point] + p2[point:]
    c2 = p2[:point] + p1[point:]
    return c1, c2

def mutate(chromosome, rate):
    new_chrom = []
    for gene in chromosome:
        if random.random() < rate:
            new_chrom.append(random.choice(LETTERS))
        else:
            new_chrom.append(gene)
    return new_chrom

# ---- Main GA Loop ----
population = [create_random_string(len(TARGET)) for _ in range(POP_SIZE)]

for gen in range(MAX_GENS):

    scores = [fitness(c) for c in population]

    best_score = max(scores)
    best_chrom = population[scores.index(best_score)]

    best_string = "".join(best_chrom)

    print("Gen", gen, "| Best:", best_string,
          "| Fitness:", best_score, "/", len(TARGET))

    if best_score == len(TARGET):
        print("*** Target matched! ***")
        break

    new_population = []
    while len(new_population) < POP_SIZE:
        p1 = select_parent(population, scores)
        p2 = select_parent(population, scores)

        if random.random() < CROSSOVER_RATE:
            c1, c2 = crossover(p1, p2)
        else:
            c1, c2 = p1[:], p2[:]

        new_population.append(mutate(c1, MUTATION_RATE))
        new_population.append(mutate(c2, MUTATION_RATE))

    population = new_population[:POP_SIZE]

Gen 0 | Best: conPZhB | Fitness: 1 / 7
Gen 1 | Best: coneqyA | Fitness: 2 / 7
Gen 2 | Best: tonPqiH | Fitness: 2 / 7
Gen 3 | Best: conehiH | Fitness: 3 / 7
Gen 4 | Best: coneDiH | Fitness: 3 / 7
Gen 5 | Best: koneQiA | Fitness: 3 / 7
Gen 6 | Best: coneQiH | Fitness: 3 / 7
Gen 7 | Best: DSnekiZ | Fitness: 3 / 7
Gen 8 | Best: coneqiA | Fitness: 3 / 7
Gen 9 | Best: TbneGiA | Fitness: 3 / 7
Gen 10 | Best: TTneCiH | Fitness: 3 / 7
Gen 11 | Best: hEneRiH | Fitness: 4 / 7
Gen 12 | Best: TSneqiH | Fitness: 3 / 7
Gen 13 | Best: hEneqiH | Fitness: 4 / 7
Gen 14 | Best: hEneqiH | Fitness: 4 / 7
Gen 15 | Best: hEneqiH | Fitness: 4 / 7
Gen 16 | Best: hEneLiB | Fitness: 4 / 7
Gen 17 | Best: hEneLiO | Fitness: 4 / 7
Gen 18 | Best: hEneLiU | Fitness: 4 / 7
Gen 19 | Best: zEneLiU | Fitness: 4 / 7
Gen 20 | Best: GineRix | Fitness: 4 / 7
Gen 21 | Best: zEnepiH | Fitness: 4 / 7
Gen 22 | Best: EEneLiU | Fitness: 4 / 7
Gen 23 | Best: REneuiQ | Fitness: 4 / 7
Gen 24 | Best: REneuic | Fitness: 5 / 7
Gen 25 | B

### Bonus: What happens if you include lowercase letters in LETTERS? Does conver-
### gence take longer? Why?
### Yes, convergence takes longer. Adding lowercase letters increases the search space, so there are more possible characters for each position. This makes it harder for the algorithm to find the correct target string, slowing down convergence.


# Reflection